# Surface-enumeration Python API

This tutorial enumerating substitutions in surface slabs using the public Python API. 

In [ ]:
from pathlib import Path
from shutil import which

from surface_pd.core import EnumerationSlab, EnumWithComposition

enumlib_available = which("enum.x") is not None and any(
    which(name) is not None
    for name in ("makestr.x", "makeStr.x", "makeStr.py")
)

The notebook expects to be started from the `examples/` directory. 

## 1. Load the slab structure

The VASP file `POSCAR_LI_100.vasp` contains an asymmetric nine-layer Li(100) slab, vacuum normal to direction 2 (the third lattice vector), and selective-dynamics flags that identify its fixed bottom region and relaxed surface. We will substitute Na for some of the Li in the surface layer. 

The surface-pd slab enumeration builds on pymatgen. The `EnumerationSlab.from_file()` factory transparently uses pymatgen to read its supported structure formats, so no separate loading step is required.

In [ ]:
structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li_100.vasp"
)
structure_path

## 2. Configure an `EnumerationSlab`

Next, we read and configure the slab in one step. `EnumerationSlab.from_file()` preserves the pymatgen structure data while adding surface-enumeration settings. Here `direction=2` selects the third lattice direction, which contains the broken periodicity and vacuum region; the lattice vector need not be perpendicular to the surface plane. `enumerated_species` selects Li, and `num_enumerated_layers={"Li": 1}` selects the outermost Li layer. Layer counts are independent for each selected species: for example, `{"Li": 1, "O": 2}` would select the outermost Li layer and two outermost O layers. Atoms with fractional coordinates in the selected direction that are within the specified `tolerance` are assigned to the same layer.

With `symmetric=False`, only the top surface is enumerated. Setting `symmetric=True` instead selects corresponding layers on both surfaces and requires a compatible symmetric slab. Our Li(100) slab model is asymmetric, and its bottom layers are fixed, so we have to disable symmetry.

In [ ]:
slab = EnumerationSlab.from_file(
    structure_path,
    direction=2,
    tolerance=0.03,
    enumerated_species=["Li"],
    num_enumerated_layers={"Li": 1},
    symmetric=False,
)

print("formula:", slab.composition.reduced_formula)
print("sites:", len(slab))
print("selective dynamics:", "selective_dynamics" in slab.site_properties)
print("vacuum-bearing lattice direction:", slab.direction)
print("selected species:", slab.enumerated_species)
print("selected layers:", slab.num_enumerated_layers)
print("symmetric enumeration:", slab.symmetric)

## 3. Inspect layers and selected sites

The `EnumerationSlab` groups sites in layers by fractional height within `tolerance`. The grouping can be inspected with the `layers_finder()` method. Additionally, `get_enumerated_site_indices()` reports exactly which original site indices may change during the enumeration, grouped by species. The method `get_fixed_region_bounds()` separately reports the bounds of the central fixed region inferred from selective-dynamics flags. Site selection does not modify the slab; at this stage, the slab is only analyzed.

In [ ]:
layers = slab.layers_finder()
selected_indices = slab.get_enumerated_site_indices()
center_bottom, center_top = slab.get_fixed_region_bounds()

print("Li layer heights:", list(layers["Li"]))
print("fixed-region bounds:", (center_bottom, center_top))
print("selected site indices:", selected_indices)

## 4. Define the ordered surface configurations

Next, we configure the enumeration. The replacement mapping below requests 50% Li to be replaced with Na. Introducing vacancies instead of other chemical species is also possible by a mapping such as `{"Li": {"Li": 0.5}}`, which would remove half of the enumerated Li atoms. 

The enumeration is done using the enumlib library and requires its binaries to be set up correctly. For each requested composition, enumlib finds symmetrically distinct orderings over the allowed surface cells. Since the surface unit cell of the original slab model only contains a single Li atom, an area multiplier of at least two is needed to realize the requested replacement (`min_cell_size=max_cell_size=2`). Larger in-plane cell multipliers permit more orderings but can greatly increase the computational effort.

At this stage, no enumeration is performed yet. We are only configuring the enumeration settings.

In [ ]:
enumerator = EnumWithComposition(
    replacements={"Li": {"Li": 0.5, "Na": 0.5}},
    min_cell_size=2,
    max_cell_size=4,
)

## 5. Run the external enumeration when requested

The enumeration is performed with the `EnumWithComposition.apply_enumeration` method, which delegates candidate generation to enumlib (via pymatgen), then returns the enumerated slab model structures as `EnumerationSlab` objects. 

The execution of the following cell is opt-in because it can take a little while to complete. Also note that enumlib is separately installed. To run the cell, install `enum.x` and `makestr.x` (or a supported `makeStr` variant), make them discoverable on `PATH`, set `RUN_ENUMLIB = True`, and rerun this cell.

In [ ]:
RUN_ENUMLIB = False
# RUN_ENUMLIB = True

if RUN_ENUMLIB and enumlib_available:
    enumerated_slabs = enumerator.apply_enumeration(
        slab, max_structures=20
    )
    print(f"enumlib returned {len(enumerated_slabs)} surface slab(s)")
    for index, result in enumerate(enumerated_slabs):
        print(index, result.formula, result.lattice.abc)
elif RUN_ENUMLIB:
    print("enumlib was requested but its executables were not found")
else:
    print("enumlib execution skipped by default; deterministic setup complete")

The following cell finds the Cartesian coordinates of the atoms in the enumerated layer of one of the returned slab models. This cell should be replaced with a simple visualization.

In [ ]:
if RUN_ENUMLIB and enumlib_available:
    top_layer_coord = list(layers["Li"])[0]
    for site in enumerated_slabs[-1].sites:
        if abs(site.frac_coords[slab.direction] - top_layer_coord) <= 5e-3:
            print(site.specie, site.coords)

## 6. Symmetric enumeration on both surfaces

Modifying the surface layer of a periodic slab model may introduce a dipole moment and lead to artificial results in electronic structure calculations without dipole correction. This is especially significant for non-metallic materials. To avoid such dipole moments, surface-pd supports symmetric enumeration, where the top and bottom surfaces of a slab model are modified in equivalent and symmetric fashion. 

Symmetric enumeration requires more than an inversion-symmetric lattice: the selective-dynamics flags must identify relaxed regions on both surfaces around a fixed central region. The one-sided Li(100) slab above is intentionally asymmetric and not suitable for a symmetric enumeration. Instead, we will use a Li$_2$O(110) slab for a symmetric enumeration example. The outermost Li layer and two outermost O layers of the Li$_2$O slab are selected independently on both surfaces.

In [ ]:
symmetric_structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li2O_110.vasp"
)
symmetric_slab = EnumerationSlab.from_file(
    symmetric_structure_path,
    direction=2,
    tolerance=0.03,
    enumerated_species=["Li", "O"],
    num_enumerated_layers={"Li": 1, "O": 2},
    symmetric=True,
)

print(
    "selected symmetric sites:",
    symmetric_slab.get_enumerated_site_indices(),
)

For this example, we will keep the Li sites fully occupied while only 75% of the selected O sites remain occupied. Surface-pd first enumerates one surface, restores complete selective-dynamics metadata, mirrors the ordering onto the second surface, and returns only finalized inversion-symmetric slabs.

In [ ]:
symmetric_enumerator = EnumWithComposition(
    replacements={"Li": {"Li": 1.0}, "O": {"O": 0.75}},
    min_cell_size=1,
    max_cell_size=2,
)

In [ ]:
RUN_SYMMETRIC_ENUMLIB = False
# RUN_SYMMETRIC_ENUMLIB = True

if RUN_SYMMETRIC_ENUMLIB and enumlib_available:
    symmetric_results = symmetric_enumerator.apply_enumeration(
        symmetric_slab, max_structures=20
    )
    print(f"enumlib returned {len(symmetric_results)} symmetric slab(s)")
    for index, result in enumerate(symmetric_results):
        print(index, result.formula, result.lattice.abc, result.is_symmetry())
elif RUN_SYMMETRIC_ENUMLIB:
    print(
        "symmetric enumlib execution requested but executables were not found"
    )
else:
    print("symmetric enumlib execution skipped by default")